# ⚡ 05. Generation 2: Quantile Gradient Boosted Trees (XGBoost)

Trains quantile regressors for $q_{0.1}, q_{0.5}, q_{0.9}$ with SHAP explanations.

In [ ]:
import xgboost as xgb
import shap

def train_and_predict_xgboost(X_train, y_train, X_test):
    val_split = int(len(X_train) * 0.9)
    X_tr, y_tr = X_train.iloc[:val_split], y_train.iloc[:val_split]
    X_val, y_val = X_train.iloc[val_split:], y_train.iloc[val_split:]
    
    preds = {}
    models = {}
    for q in [0.1, 0.5, 0.9]:
        model = xgb.XGBRegressor(
            objective='reg:quantileerror',
            quantile_alpha=q,
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.75,
            early_stopping_rounds=50,
            tree_method='hist',
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        preds[q] = model.predict(X_test)
        models[q] = model
        
    # Explain median model using SHAP
    explainer = shap.TreeExplainer(models[0.5])
    shap_vals = explainer.shap_values(X_test.iloc[:min(100, len(X_test))])
    
    return preds[0.1], preds[0.5], preds[0.9], models[0.5]
